# 매출 정리

### 추정 매출
1. 각 상권별 5분위
2. 전체 매장에서의 5분위 

1번은 상권 맵핑 (완)/2번은 번화가의 위치 요인이 크게 작용할 우려가 있음


**2번 먼저 실행하기**

In [ ]:
import pandas as pd

# 데이터 불러오기
df_sales = pd.read_csv(r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\점수산출\cafes_estimated_sales.csv')
df_tradar = pd.read_csv(r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\상권별_카페_점포_수.csv', encoding = 'euc-kr')
print(df_sales['cafes_sales'].isna().value_counts())
# 필요한 컬럼 추출 및 이름 정리
s_code = df_sales[['area_code', 'cafes_sales']].copy()
t_code = df_tradar[['TRDAR_CD', 'TRDAR_CD_N']].copy()
t_code.rename(columns={'TRDAR_CD': 'area_code'}, inplace=True)

# 병합 (왼쪽 기준: 상권 전체 목록)
df_merge = pd.merge(t_code, s_code, how='left', on='area_code')

# 매출 정보가 없는 상권 찾기
missing_sales_areas = df_merge[df_merge['cafes_sales'].isna()]

# 결과 출력
print(missing_sales_areas[['area_code', 'TRDAR_CD_N']])

# 생각보다 매출 데이터를 사용할 수 없는 곳이 많음: 매장 8734개

cafes_sales
False    17237
True      1036
Name: count, dtype: int64
       area_code                    TRDAR_CD_N
141      3110001                        이북5도청사
142      3110001                        이북5도청사
151      3110004                        대신고등학교
318      3110025                         충신시장옆
319      3110025                         충신시장옆
...          ...                           ...
18268    3130327  평화시장(남평화시장, 제일평화시장, 신평화패션타운)
18269    3130327  평화시장(남평화시장, 제일평화시장, 신평화패션타운)
18270    3130327  평화시장(남평화시장, 제일평화시장, 신평화패션타운)
18271    3130327  평화시장(남평화시장, 제일평화시장, 신평화패션타운)
18272    3130327  평화시장(남평화시장, 제일평화시장, 신평화패션타운)

[1036 rows x 2 columns]


In [18]:
# 결측치 제외, 전체 매장 5분위 실시, 점수 부여 나머지는 우선 그냥 둬 봐
import pandas as pd
df_sales = pd.read_csv(r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\점수산출\cafes_estimated_sales.csv')

# 점수 계산 함수
def assign_review_score(series):
    # 기본값을 NaN으로 초기화 (NaN은 그대로 유지되도록)
    score = pd.Series(index=series.index, dtype="float")

    # 0인 항목은 0점
    score[(series == 0)] = 0

    # 0 초과 & notna 값만 대상으로 분위수 기반 점수 부여
    valid_mask = (series > 0) & (series.notna())
    valid_series = series[valid_mask]

    # 분위수 기반 구간 나누기 (labels: 1~5, 높은 수치에 높은 점수)
    quantiles = pd.qcut(valid_series, q=5, labels=[1, 2, 3, 4, 5])

    # 결과 반영
    score[valid_mask] = quantiles.astype(int)

    return score

# 적용
df_sales['cafes_sales'] = pd.to_numeric(df_sales['cafes_sales'], errors='coerce')
df_sales['sales_score_total'] = assign_review_score(df_sales['cafes_sales'])
print(df_sales[['상가업소번호','상호명','cafes_sales','sales_score_total']].head(10))

                 상가업소번호      상호명   cafes_sales  sales_score_total
0  MA010120220806361295  000케이크바  1.201659e+08                3.0
1  MA0101202406A0096622  0122커피바  4.221296e+07                1.0
2  MA0101202409A0196345  0125커피바  1.006692e+07                1.0
3  MA0101202306A0062257  025베이커리           NaN                NaN
4  MA0101202406A0132263     09커피  3.012311e+07                1.0
5  MA0101202209A0088428  1%리버티커피  3.283196e+08                5.0
6  MA010120220808588994    1.5도씨  5.089181e+07                2.0
7  MA010120220811909716   100X커피  1.306042e+08                3.0
8  MA0101202305A0121922   100디그리  2.042075e+08                4.0
9  MA010120220808810840   101빌리지  3.113676e+07                1.0


In [24]:
# 기존 점수랑 합치기
import pandas as pd
df_score = pd.read_csv(r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\점수산출\카페_점수.csv')
df_sale = df_sales[['상가업소번호','sales_score_total']].copy()
df_merged = pd.merge(df_score, df_sale, on = '상가업소번호', how = 'left')
df_merged.to_csv(r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\점수산출\카페_점수.csv', encoding = 'utf-8-sig',index = False)

**1번 실행하기**

In [36]:
#eda
df_sales = pd.read_csv(r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\점수산출\cafes_estimated_sales.csv')
unique_sales_per_area = df_sales.groupby('area_code')['cafes_sales'].nunique()

# 모든 값이 같은 상권만 보기
same_sales_area = unique_sales_per_area[unique_sales_per_area == 1]
print(same_sales_area.value_counts())



cafes_sales
1    1110
Name: count, dtype: int64


In [27]:
import pandas as pd
# 상권별로 나눈 다음에 계산
def assign_grouped_review_score(df, group_col, value_col):
    def score_within_group(group):
        score = pd.Series(index=group.index, dtype='float')

        # 0점 처리
        score[group[value_col] == 0] = 0

        # 0 초과 & notna인 값만 추출
        valid_mask = (group[value_col] > 0) & (group[value_col].notna())
        valid_values = group.loc[valid_mask, value_col]

        if len(valid_values) >= 5:
            try:
                # 분위수 계산 시 중복 구간 제거
                quantiles = pd.qcut(valid_values, q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')
                score[valid_mask] = quantiles.astype(int)
            except ValueError:
                # 값이 모두 동일해서 실패하면 3점 부여
                score[valid_mask] = 3
        else:
            score[valid_mask] = 3

        return score

    return df.groupby(group_col).apply(score_within_group, include_groups=False).reset_index(level=0, drop=True)

# 결측치 제외, 상권별 실시

df_sales = pd.read_csv(r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\점수산출\cafes_estimated_sales.csv')
# 적용
df_sales['cafes_sales'] = pd.to_numeric(df_sales['cafes_sales'], errors='coerce')

df_sales['sales_score_grouped'] = assign_grouped_review_score(
    df_sales,
    group_col='area_code',
    value_col='cafes_sales'
)

print(df_sales[['상가업소번호','상호명','cafes_sales','sales_score_grouped']].head(10))


                 상가업소번호      상호명   cafes_sales  sales_score_grouped
0  MA010120220806361295  000케이크바  1.201659e+08                  3.0
1  MA0101202406A0096622  0122커피바  4.221296e+07                  3.0
2  MA0101202409A0196345  0125커피바  1.006692e+07                  3.0
3  MA0101202306A0062257  025베이커리           NaN                  NaN
4  MA0101202406A0132263     09커피  3.012311e+07                  3.0
5  MA0101202209A0088428  1%리버티커피  3.283196e+08                  3.0
6  MA010120220808588994    1.5도씨  5.089181e+07                  3.0
7  MA010120220811909716   100X커피  1.306042e+08                  3.0
8  MA0101202305A0121922   100디그리  2.042075e+08                  3.0
9  MA010120220808810840   101빌리지  3.113676e+07                  3.0


In [28]:
# 기존 점수랑 합치기
import pandas as pd
df_score = pd.read_csv(r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\점수산출\카페_점수.csv')
df_sale = df_sales[['상가업소번호','sales_score_grouped']].copy()
df_merged = pd.merge(df_score, df_sale, on = '상가업소번호', how = 'left')
df_merged.to_csv(r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\점수산출\카페_점수.csv', encoding = 'utf-8-sig',index = False)